# Cross-Domain Generalization Analysis

Identifies jargon-sensitive components that generalize across domains
(Section 5.4, Table 5 of the paper).

The analysis compares per-component accuracy across three tasks:
- **Med-JI** (`jargon_detect`): Medical jargon identification — the primary task
- **Mat-JI** (`matsci_jargon_detect`): Materials science jargon identification — cross-domain transfer
- **BoolQ** (`boolq_control`): General reading comprehension — non-jargon control

Components that rank highly on both jargon tasks but NOT on BoolQ are
interpreted as encoding a partially domain-agnostic notion of specialized
terminology, rather than just general A/B classification competence.

**Prerequisites**: Before running this notebook, generate the Mat-JI dataset
and run `decompose.py` (from [terarachang/LLMDecomp](https://github.com/terarachang/LLMDecomp))
for all three datasets. See the commands below in the Setup section.

In [2]:
from datasets import load_dataset
import os
from pathlib import Path
import pandas as pd
from datasets import Dataset
from tqdm import tqdm
import json
import numpy as np

/home-nfs/dkeng/myenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import os
os.chdir('NLP4MatSci-ACL23')  # or 'LLMDecomp-main' if from zip
print(f"Working directory: {os.getcwd()}")

Working directory: /home-nfs/dkeng/NLP4MatSci-ACL23


## Cross-task comparison table

Builds a full component-level comparison table (top-K components ranked
by Med-JI accuracy, with Mat-JI and BoolQ accuracy shown alongside).

In [5]:
from scipy.stats import entropy
import pandas as pd
import numpy as np
import torch
import os

# ---------------------------------------------------------------------------
# Paths — update these to match your local environment
# ---------------------------------------------------------------------------
MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B-Instruct"
N_SHOTS = 0
FORMAT = 1
SEED = 0

# Datasets
PRIMARY_DATASET = "jargon_detect"           # Top components selected from here
COMPARE_DATASETS = ["boolq_control", "matsci_jargon_detect"]  # Compare against these

ALL_DATASETS = [PRIMARY_DATASET] + COMPARE_DATASETS
DATASET_LABELS = {
    "matsci_jargon_detect": "matsci_jargon_detect",
    "boolq_control": "BoolQ",
    "jargon_detect": "jargon_detect",
}

TOP_K = 20

# ===== LOAD ALL DATASETS =====
data = {}

for dataset in ALL_DATASETS:
    dir_path = f"DECOMP/{MODEL_NAME}/{dataset}/{N_SHOTS}shots/f{FORMAT}"
    label = DATASET_LABELS.get(dataset, dataset)
    print(f"Loading {label} ({dataset})")
    print(f"  Dir: {dir_path}")

    acc_heads = np.load(os.path.join(dir_path, f'acc-heads-test-{SEED}.npy'))
    acc_mlps = np.load(os.path.join(dir_path, f'acc-mlps-test-{SEED}.npy'))
    acc_full = np.load(os.path.join(dir_path, f'acc-resid_post-test-{SEED}.npy'))
    projs_heads = torch.load(os.path.join(dir_path, f'projs-heads-test-{SEED}.pt'))
    projs_mlps = torch.load(os.path.join(dir_path, f'projs-mlps-test-{SEED}.pt'))

    n_options = projs_heads.shape[-1]
    print(f"  Full model accuracy: {acc_full:.2%}, Options: {n_options}\n")

    data[dataset] = {
        'acc_heads': acc_heads,
        'acc_mlps': acc_mlps,
        'acc_full': acc_full,
        'projs_heads': projs_heads,
        'projs_mlps': projs_mlps,
        'n_options': n_options,
    }

n_layers, n_heads = data[PRIMARY_DATASET]['acc_heads'].shape
print(f"Model: {n_layers} layers, {n_heads} heads per layer\n")

# ===== COMPUTE ENTROPY =====
def compute_entropy(predictions, n_options):
    counts = np.bincount(predictions, minlength=n_options)
    probs = counts / len(predictions)
    return entropy(probs, base=2)

def compute_all_entropy(projs_heads, projs_mlps, n_layers, n_heads, n_options):
    head_ent = np.zeros((n_layers, n_heads))
    for l in range(n_layers):
        for h in range(n_heads):
            preds = projs_heads[l, h].argmax(dim=-1).numpy()
            head_ent[l, h] = compute_entropy(preds, n_options)

    mlp_ent = np.zeros(n_layers)
    for l in range(n_layers):
        preds = projs_mlps[l].argmax(dim=-1).numpy()
        mlp_ent[l] = compute_entropy(preds, n_options)

    return head_ent, mlp_ent

for dataset in ALL_DATASETS:
    d = data[dataset]
    head_ent, mlp_ent = compute_all_entropy(
        d['projs_heads'], d['projs_mlps'], n_layers, n_heads, d['n_options']
    )
    d['head_entropy'] = head_ent
    d['mlp_entropy'] = mlp_ent

# ===== COLLECT ALL COMPONENTS =====
all_components = []

for l in range(n_layers):
    for h in range(n_heads):
        row = {
            'Type': 'Head',
            'Layer': l,
            'Head/MLP': h,
        }
        for dataset in ALL_DATASETS:
            label = DATASET_LABELS[dataset]
            d = data[dataset]
            row[f'Acc_{label}'] = d['acc_heads'][l, h]
            row[f'Ent_{label}'] = d['head_entropy'][l, h]
        all_components.append(row)

for l in range(n_layers):
    row = {
        'Type': 'MLP',
        'Layer': l,
        'Head/MLP': 'MLP',
    }
    for dataset in ALL_DATASETS:
        label = DATASET_LABELS[dataset]
        d = data[dataset]
        row[f'Acc_{label}'] = d['acc_mlps'][l]
        row[f'Ent_{label}'] = d['mlp_entropy'][l]
    all_components.append(row)

df = pd.DataFrame(all_components)

# ===== COMPUTE PER-TASK RANKS (1 = best) =====
primary_label = DATASET_LABELS[PRIMARY_DATASET]

for dataset in ALL_DATASETS:
    label = DATASET_LABELS[dataset]
    # rank with method='min' so ties get the same rank; ascending=False so highest acc = rank 1
    df[f'Rank_{label}'] = df[f'Acc_{label}'].rank(ascending=False, method='min').astype(int)

total_components = len(df)

# Sort by primary task accuracy, take top K
df_sorted = df.sort_values(f'Acc_{primary_label}', ascending=False).reset_index(drop=True)
df_top = df_sorted.head(TOP_K).copy()

# Compute accuracy differences vs primary
for dataset in COMPARE_DATASETS:
    label = DATASET_LABELS[dataset]
    df_top[f'Δ_{label}'] = df_top[f'Acc_{label}'] - df_top[f'Acc_{primary_label}']

# ===== PRINT TABLE =====
print("=" * 180)
print(f"TOP {TOP_K} COMPONENTS BY {primary_label.upper()} ACCURACY → CROSS-TASK COMPARISON (out of {total_components} total components)")
print("=" * 180)

# Header
header = (f"{'':>4} | {'Type':<4} | {'Lay':>3} | {'Head':>4} |")
for dataset in ALL_DATASETS:
    label = DATASET_LABELS[dataset]
    header += f" {label:>8} {'Ent':>6} {'Rank':>5} |"
for dataset in COMPARE_DATASETS:
    label = DATASET_LABELS[dataset]
    header += f" {'Δ '+label:>10} |"
print(header)
print("-" * 180)

for idx, row in df_top.iterrows():
    rank = idx + 1
    type_str = row['Type']
    layer_str = f"{row['Layer']:2d}"
    head_str = f"{row['Head/MLP']:2d}" if row['Type'] == 'Head' else "MLP"

    line = f" {rank:2d}  | {type_str:<4} | {layer_str:>3} | {head_str:>4} |"

    for dataset in ALL_DATASETS:
        label = DATASET_LABELS[dataset]
        acc = f"{row[f'Acc_{label}']:.2%}"
        ent = f"{row[f'Ent_{label}']:.3f}"
        rnk = f"{row[f'Rank_{label}']:d}"
        line += f" {acc:>8} {ent:>6} {rnk:>5} |"

    for dataset in COMPARE_DATASETS:
        label = DATASET_LABELS[dataset]
        diff = row[f'Δ_{label}']
        line += f" {diff:>+10.2%} |"

    print(line)

# Full model row
print("-" * 180)
full_line = f"     | FULL |     |      |"
for dataset in ALL_DATASETS:
    label = DATASET_LABELS[dataset]
    acc = f"{data[dataset]['acc_full']:.2%}"
    full_line += f" {acc:>8} {'':>6} {'':>5} |"
for dataset in COMPARE_DATASETS:
    label = DATASET_LABELS[dataset]
    diff = data[dataset]['acc_full'] - data[PRIMARY_DATASET]['acc_full']
    full_line += f" {diff:>+10.2%} |"
print(full_line)
print("=" * 180)

# ===== SUMMARY STATS =====
print(f"\nSUMMARY FOR TOP {TOP_K} COMPONENTS:")
print("=" * 180)

# Average accuracies and ranks
for dataset in ALL_DATASETS:
    label = DATASET_LABELS[dataset]
    avg_acc = df_top[f'Acc_{label}'].mean()
    avg_rank = df_top[f'Rank_{label}'].mean()
    median_rank = df_top[f'Rank_{label}'].median()
    print(f"  {label:>10}:  Avg Acc = {avg_acc:.2%}  |  Avg Rank = {avg_rank:.1f}  |  Median Rank = {median_rank:.0f}  (out of {total_components})")

print()

# Per compare dataset stats
for dataset in COMPARE_DATASETS:
    label = DATASET_LABELS[dataset]
    acc_full_cmp = data[dataset]['acc_full']

    avg_diff = df_top[f'Δ_{label}'].mean()
    above_full = (df_top[f'Acc_{label}'] > acc_full_cmp).sum()
    big_drop = (df_top[f'Δ_{label}'] < -0.10).sum()
    big_gain = (df_top[f'Δ_{label}'] > 0.10).sum()
    # How many stay in the top K on this task too
    stay_top_k = (df_top[f'Rank_{label}'] <= TOP_K).sum()

    print(f"  {label}:")
    print(f"    Avg Δ accuracy:                    {avg_diff:+.2%}")
    print(f"    Components above full-model:        {above_full}/{TOP_K}")
    print(f"    Components also in top {TOP_K}:          {stay_top_k}/{TOP_K}")
    print(f"    Components with >10pp drop:         {big_drop}/{TOP_K}")
    print(f"    Components with >10pp gain:         {big_gain}/{TOP_K}")
    print()

print("=" * 180)

# ===== SAVE =====
compare_names = "_vs_".join(DATASET_LABELS[d] for d in COMPARE_DATASETS)
output_file = f"top{TOP_K}_{PRIMARY_DATASET}_vs_{compare_names}_seed{SEED}.csv"
df_top.to_csv(output_file, index=False)
print(f"\n✓ Saved to: {output_file}")

Loading jargon_detect (jargon_detect)
  Dir: DECOMP/meta-llama/Meta-Llama-3.1-8B-Instruct/jargon_detect/0shots/f1
  Full model accuracy: 58.80%, Options: 2

Loading BoolQ (boolq_control)
  Dir: DECOMP/meta-llama/Meta-Llama-3.1-8B-Instruct/boolq_control/0shots/f1
  Full model accuracy: 81.55%, Options: 2

Loading matsci_jargon_detect (matsci_jargon_detect)
  Dir: DECOMP/meta-llama/Meta-Llama-3.1-8B-Instruct/matsci_jargon_detect/0shots/f1
  Full model accuracy: 73.85%, Options: 2

Model: 32 layers, 32 heads per layer

TOP 20 COMPONENTS BY JARGON_DETECT ACCURACY → CROSS-TASK COMPARISON (out of 1056 total components)
     | Type | Lay | Head | jargon_detect    Ent  Rank |    BoolQ    Ent  Rank | matsci_jargon_detect    Ent  Rank |    Δ BoolQ | Δ matsci_jargon_detect |
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
  1  | Head |  25 |    8 |   60.60%  0.912  

## Jargon-specific component filter (Section 5.4, Table 5)

Identifies components that:
1. Rank in the top-`TOP_K` on **both** Med-JI and Mat-JI
2. Do **not** rank in the top-`TOP_K` on BoolQ

This filter isolates components whose strong jargon performance is not
explained by general A/B classification competence (BoolQ is also a
binary A/B task, so any component that simply favors one letter would
score highly on all three).

In [4]:
import numpy as np
from scipy.stats import spearmanr, weightedtau

# Update path to your local path
# Load the accuracy arrays
heads_med = np.load('./DECOMP/meta-llama/Meta-Llama-3.1-8B-Instruct/jargon_detect/0shots/f1/acc-heads-test-0.npy')   # shape (32, 32) - medical LLM
heads_mat = np.load('./DECOMP/meta-llama/Meta-Llama-3.1-8B-Instruct/matsci_jargon_detect/0shots/f1/acc-heads-test-0.npy')     # shape (32, 32) - general LLM
heads_boolq = np.load('./DECOMP/meta-llama/Meta-Llama-3.1-8B-Instruct/boolq_control/0shots/f1/acc-heads-test-0.npy')
mlps_med  = np.load('./DECOMP/meta-llama/Meta-Llama-3.1-8B-Instruct/jargon_detect/0shots/f1/acc-mlps-test-0.npy')    # shape (32,)
mlps_mat  = np.load('./DECOMP/meta-llama/Meta-Llama-3.1-8B-Instruct/matsci_jargon_detect/0shots/f1/acc-mlps-test-0.npy')    # shape (32,)
mlps_boolq  = np.load('./DECOMP/meta-llama/Meta-Llama-3.1-8B-Instruct/boolq_control/0shots/f1/acc-mlps-test-0.npy')  
# Flatten heads to 1D so each entry corresponds to one (layer, head) component
heads_med_flat = heads_med.flatten()   # 1024 components
heads_mat_flat = heads_mat.flatten()
heads_boolq_flat = heads_boolq.flatten()
mlps_med_flat = mlps_med.flatten()
mlps_mat_flat = mlps_mat.flatten()
mlps_boolq_flat = mlps_boolq.flatten()

# Combined (heads + MLPs together) as a single ranking of all components
all_med = np.concatenate([heads_med_flat, mlps_med])
all_mat = np.concatenate([heads_mat_flat, mlps_mat])
all_boolq = np.concatenate([heads_boolq_flat, mlps_boolq])

In [8]:
import numpy as np

def component_name(i):
    if i < 1024:
        return f"L{i // 32}H{i % 32}"
    else:
        return f"MLP-{i - 1024}"

# --- Approach 1: Set-based filter (top-k on jargon, NOT top-k on BoolQ) ---
def jargon_specific_topk(med, mat, boolq, k_jargon=50, k_boolq=50):
    """Components in top-k of both jargon tasks but NOT in top-k of BoolQ."""
    top_med = set(np.argsort(-med)[:k_jargon])
    top_mat = set(np.argsort(-mat)[:k_jargon])
    top_boolq = set(np.argsort(-boolq)[:k_boolq])
    
    jargon_shared = top_med & top_mat
    jargon_specific = jargon_shared - top_boolq
    return jargon_specific, jargon_shared

specific, shared = jargon_specific_topk(all_med, all_mat, all_boolq, k_jargon=50, k_boolq=50)
print(f"Top-50 shared by both jargon tasks: {len(shared)}")
print(f"Of those, NOT in BoolQ top-50: {len(specific)}")
print()
print("Jargon-specific components (high on Med-JI AND Mat-JI, low on BoolQ):")
for i in sorted(specific, key=lambda x: -(all_med[x] + all_mat[x])):
    print(f"  {component_name(i):>10}  Med={all_med[i]:.3f}  Mat={all_mat[i]:.3f}  BoolQ={all_boolq[i]:.3f}")


Top-50 shared by both jargon tasks: 35
Of those, NOT in BoolQ top-50: 9

Jargon-specific components (high on Med-JI AND Mat-JI, low on BoolQ):
      L20H10  Med=0.562  Mat=0.769  BoolQ=0.505
      L18H26  Med=0.553  Mat=0.735  BoolQ=0.539
      L29H16  Med=0.532  Mat=0.673  BoolQ=0.533
      MLP-19  Med=0.598  Mat=0.595  BoolQ=0.532
       L19H5  Med=0.571  Mat=0.605  BoolQ=0.349
      L15H11  Med=0.552  Mat=0.619  BoolQ=0.531
      L21H18  Med=0.578  Mat=0.589  BoolQ=0.501
      L26H27  Med=0.570  Mat=0.590  BoolQ=0.510
       L26H4  Med=0.537  Mat=0.607  BoolQ=0.448
